# U2T01: Task 3 - Part-of-Speech Tagging (UD English EWT)
## Adaptation Ladder: Feature-Based Adaptation vs. Partial Fine-Tuning on bert-base-uncased

### Context & Objectives
This notebook adapts `bert-base-uncased` to the **Universal Dependencies English EWT** (UD English EWT) benchmark across 17 universal POS (UPOS) tags.

In accordance with the assignment requirements:
1. **Token-level subword handling:** Words split into subtokens assign the ground-truth label strictly to the **first subword**. Subsequent subwords, `[CLS]`, `[SEP]`, and padding tokens are masked with `-100` (ignored by `CrossEntropyLoss`).
2. **Mandatory sanity-check:** Pre-training batch inspection printing tokens alongside their aligned UPOS labels.
3. **Two adaptation methods compared:**
   - **Experiment A (Feature-based adaptation):** Frozen BERT body (0 BERT parameters updated) + linear classifier per token over hidden states.
   - **Experiment B (Partial Fine-Tuning):** Frozen layers 0 to 9; top 2 encoder layers (10 and 11, ~14M params) and classification head are trained.
4. **Differential learning rates:** Head: `1e-3`, Backbone top layers: `2e-5`.
5. **Metrics tracked:** Per-token Accuracy (excluding `-100`), Macro-F1 across all 17 UPOS tags, loss, and training wall-clock time.


### 1. Environment Setup & Reproducibility
Run this cell in Google Colab to install dependencies and configure GPU acceleration.


In [ ]:
# Install dependencies in Colab
!pip install -q transformers datasets evaluate accelerate scikit-learn matplotlib huggingface_hub

import os
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForTokenClassification,
    get_linear_schedule_with_warmup,
    DataCollatorForTokenClassification
)
from datasets import load_dataset
import evaluate

# Set fixed seeds for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")


### 2. Dataset Loading & UPOS Tag Definitions
We load the UD English EWT dataset (`universal-dependencies/universal_dependencies`, config `en_ewt`).
There are 17 universal POS tags (ADJ, ADP, ADV, AUX, CCONJ, DET, INTJ, NOUN, NUM, PART, PRON, PROPN, PUNCT, SCONJ, SYM, VERB, X).


In [ ]:
# Load Universal Dependencies English EWT
raw_dataset = load_dataset("universal-dependencies/universal_dependencies", "en_ewt")
print(raw_dataset)

upos_feature = raw_dataset["train"].features["upos"]
label_names = upos_feature.feature.names
id2label = {i: label for i, label in enumerate(label_names)}
label2id = {label: i for i, label in enumerate(label_names)}

print(f"Number of UPOS classes: {len(label_names)}")
print("UPOS Tags:", label_names)

print("\nSample training sentence:")
sample = raw_dataset["train"][0]
print("Tokens:", sample["tokens"][:10])
print("UPOS:", [label_names[t] for t in sample["upos"][:10]])


### 3. Subword Tokenization & Alignment
- First subword gets the UPOS tag.
- Continuation subwords get `-100`.
- `[CLS]`, `[SEP]`, padding get `-100`.


In [ ]:
model_ckpt = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

def tokenize_and_align_pos(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=128
    )

    labels = []
    for i, label in enumerate(examples["upos"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Tokenize full dataset splits
tokenized_datasets = raw_dataset.map(
    tokenize_and_align_pos,
    batched=True,
    remove_columns=raw_dataset["train"].column_names
)
print("POS tokenization and alignment finished.")


### 4. Mandatory Sanity Check: Inspect Token-Label Alignments
Printing a sample batch showing tokens with their corresponding UPOS tags or `[IGNORE: -100]`.


In [ ]:
example = tokenized_datasets["train"][0]
tokens = tokenizer.convert_ids_to_tokens(example["input_ids"])
labels = example["labels"]

print(f"{'INDEX':<6} {'TOKEN':<18} {'TOKEN ID':<10} {'LABEL ID':<10} {'UPOS TAG'}")
print("-" * 65)
for idx, (token, token_id, label_id) in enumerate(zip(tokens, example["input_ids"], labels)):
    tag_str = id2label[label_id] if label_id != -100 else "[IGNORE: -100]"
    print(f"{idx:<6} {token:<18} {token_id:<10} {label_id:<10} {tag_str}")

print("-" * 65)
print("Sanity check verified successfully.")


### 5. Data Loaders & Evaluation Metrics
We measure Accuracy and Macro-F1 across all valid tokens (ignoring `-100`).


In [ ]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

batch_size = 32
train_loader = DataLoader(tokenized_datasets["train"], batch_size=batch_size, shuffle=True, collate_fn=data_collator)
val_loader = DataLoader(tokenized_datasets["validation"], batch_size=batch_size, shuffle=False, collate_fn=data_collator)
test_loader = DataLoader(tokenized_datasets["test"], batch_size=batch_size, shuffle=False, collate_fn=data_collator)

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def evaluate_pos(model, data_loader, device):
    model.eval()
    all_preds = []
    all_labels = []
    total_loss = 0.0

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            logits = outputs.logits

            total_loss += loss.item() * input_ids.size(0)

            predictions = torch.argmax(logits, dim=-1).cpu().numpy()
            labels_np = labels.cpu().numpy()

            # Filter out -100
            for pred_seq, label_seq in zip(predictions, labels_np):
                for p, l in zip(pred_seq, label_seq):
                    if l != -100:
                        all_preds.append(p)
                        all_labels.append(l)

    avg_loss = total_loss / len(data_loader.dataset)
    acc = accuracy_metric.compute(predictions=all_preds, references=all_labels)["accuracy"]
    macro_f1 = f1_metric.compute(predictions=all_preds, references=all_labels, average="macro")["f1"]
    return avg_loss, acc, macro_f1


### 6. Experiment A: Feature-Based Adaptation (Frozen BERT)
**Concept:**
- Freeze all 110M parameters of BERT.
- Train only a linear token classification head on top of the contextual representations.
- Trainable parameters: ~13k (0 BERT parameters).


In [ ]:
class BertFeatureTokenClassifier(nn.Module):
    def __init__(self, model_name="bert-base-uncased", num_labels=17):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        for param in self.bert.parameters():
            param.requires_grad = False
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)
        self.loss_fn = nn.CrossEntropyLoss(ignore_index=-100)

    def forward(self, input_ids, attention_mask, labels=None):
        with torch.no_grad():
            outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
            sequence_output = outputs.last_hidden_state

        sequence_output = self.dropout(sequence_output)
        logits = self.classifier(sequence_output)

        loss = None
        if labels is not None:
            loss = self.loss_fn(logits.view(-1, logits.size(-1)), labels.view(-1))

        class Output:
            def __init__(self, loss, logits):
                self.loss = loss
                self.logits = logits
        return Output(loss, logits)

model_feature = BertFeatureTokenClassifier(model_ckpt, num_labels=len(label_names)).to(device)
trainable_params_a = sum(p.numel() for p in model_feature.parameters() if p.requires_grad)
total_params_a = sum(p.numel() for p in model_feature.parameters())
print(f"[Experiment A - Feature-based] Trainable: {trainable_params_a:,} / {total_params_a:,} ({trainable_params_a/total_params_a*100:.3f}%)")

optimizer_a = AdamW(model_feature.classifier.parameters(), lr=1e-3, weight_decay=0.01)
epochs = 3
history_a = {"train_loss": [], "val_loss": [], "val_acc": [], "val_f1": [], "epoch_times": []}

print("\n--- Starting Training: Experiment A (Feature-based) ---")
for epoch in range(epochs):
    t0 = time.time()
    model_feature.train()
    total_train_loss = 0.0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer_a.zero_grad()
        out = model_feature(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = out.loss
        loss.backward()
        optimizer_a.step()

        total_train_loss += loss.item() * input_ids.size(0)

    epoch_time = time.time() - t0
    train_loss = total_train_loss / len(train_loader.dataset)
    val_loss, val_acc, val_f1 = evaluate_pos(model_feature, val_loader, device)

    history_a["train_loss"].append(train_loss)
    history_a["val_loss"].append(val_loss)
    history_a["val_acc"].append(val_acc)
    history_a["val_f1"].append(val_f1)
    history_a["epoch_times"].append(epoch_time)

    print(f"Epoch {epoch+1}/{epochs} | Time: {epoch_time:.2f}s | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | Macro-F1: {val_f1:.4f}")


### 7. Experiment B: Partial Fine-Tuning (Top 2 Encoder Layers + Head)
**Concept:**
- Freeze embeddings and encoder layers 0 to 9.
- Train only top 2 encoder layers (10 and 11) + token classification head (~14M parameters).
- **Differential Learning Rates:**
  - Head: `1e-3`
  - Encoder top layers: `2e-5`


In [ ]:
model_partial = AutoModelForTokenClassification.from_pretrained(
    model_ckpt,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id
).to(device)

# Freeze embeddings and layers 0 through 9
for param in model_partial.bert.embeddings.parameters():
    param.requires_grad = False
for layer in model_partial.bert.encoder.layer[:10]:
    for param in layer.parameters():
        param.requires_grad = False

head_params_partial = []
top_layer_params_partial = []

for name, param in model_partial.named_parameters():
    if not param.requires_grad:
        continue
    if "classifier" in name:
        head_params_partial.append(param)
    else:
        top_layer_params_partial.append(param)

optimizer_partial = AdamW([
    {"params": top_layer_params_partial, "lr": 2e-5, "weight_decay": 0.01},
    {"params": head_params_partial, "lr": 1e-3, "weight_decay": 0.01}
])

trainable_params_b = sum(p.numel() for p in model_partial.parameters() if p.requires_grad)
print(f"[Experiment B - Partial FT] Trainable: {trainable_params_b:,} / {total_params_a:,} ({trainable_params_b/total_params_a*100:.2f}%)")

total_steps_b = len(train_loader) * epochs
scheduler_partial = get_linear_schedule_with_warmup(optimizer_partial, num_warmup_steps=int(total_steps_b*0.1), num_training_steps=total_steps_b)

history_b = {"train_loss": [], "val_loss": [], "val_acc": [], "val_f1": [], "epoch_times": []}

print("\n--- Starting Training: Experiment B (Partial Fine-Tuning) ---")
for epoch in range(epochs):
    t0 = time.time()
    model_partial.train()
    total_train_loss = 0.0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer_partial.zero_grad()
        outputs = model_partial(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        nn.utils.clip_grad_norm_(model_partial.parameters(), 1.0)
        optimizer_partial.step()
        scheduler_partial.step()

        total_train_loss += loss.item() * input_ids.size(0)

    epoch_time = time.time() - t0
    train_loss = total_train_loss / len(train_loader.dataset)
    val_loss, val_acc, val_f1 = evaluate_pos(model_partial, val_loader, device)

    history_b["train_loss"].append(train_loss)
    history_b["val_loss"].append(val_loss)
    history_b["val_acc"].append(val_acc)
    history_b["val_f1"].append(val_f1)
    history_b["epoch_times"].append(epoch_time)

    print(f"Epoch {epoch+1}/{epochs} | Time: {epoch_time:.2f}s | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | Macro-F1: {val_f1:.4f}")


### 8. Evaluation on Test Set & Comparative Analysis
We evaluate both models on the held-out test split of UD English EWT.


In [ ]:
_, test_acc_a, test_f1_a = evaluate_pos(model_feature, test_loader, device)
_, test_acc_b, test_f1_b = evaluate_pos(model_partial, test_loader, device)

comparison_df = pd.DataFrame({
    "Method": ["Feature-Based (Frozen BERT)", "Partial Fine-Tuning (Top 2 Layers)"],
    "Trainable Params": [f"{trainable_params_a:,}", f"{trainable_params_b:,}"],
    "Total Train Time (s)": [f"{sum(history_a['epoch_times']):.2f}", f"{sum(history_b['epoch_times']):.2f}"],
    "Avg Sec/Epoch": [f"{np.mean(history_a['epoch_times']):.2f}", f"{np.mean(history_b['epoch_times']):.2f}"],
    "Test Token Accuracy": [f"{test_acc_a:.4f}", f"{test_acc_b:.4f}"],
    "Test Macro-F1": [f"{test_f1_a:.4f}", f"{test_f1_b:.4f}"],
    "Status": ["Rejected Alternative", "Delivered Model"]
})

print("=== UD ENGLISH EWT POS COMPARISON SUMMARY ===")
display(comparison_df)

# Visualizations
epochs_range = range(1, epochs + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss Curves
axes[0].plot(epochs_range, history_a["train_loss"], 'o--', label="Train Loss (Feature-based)", color="green")
axes[0].plot(epochs_range, history_a["val_loss"], 'o-', label="Val Loss (Feature-based)", color="lime")
axes[0].plot(epochs_range, history_b["train_loss"], 's--', label="Train Loss (Partial FT)", color="indigo")
axes[0].plot(epochs_range, history_b["val_loss"], 's-', label="Val Loss (Partial FT)", color="mediumpurple")
axes[0].set_title("Training & Validation Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross-Entropy Loss")
axes[0].legend()
axes[0].grid(True, linestyle="--", alpha=0.5)

# Accuracy & F1 Curves
axes[1].plot(epochs_range, history_a["val_acc"], 'o-', label="Val Acc (Feature-based)", color="green")
axes[1].plot(epochs_range, history_b["val_acc"], 's-', label="Val Acc (Partial FT)", color="indigo")
axes[1].plot(epochs_range, history_a["val_f1"], 'o--', label="Macro-F1 (Feature-based)", color="lime")
axes[1].plot(epochs_range, history_b["val_f1"], 's--', label="Macro-F1 (Partial FT)", color="mediumpurple")
axes[1].set_title("Validation Accuracy & Macro-F1")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Score")
axes[1].legend()
axes[1].grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig("task3_pos_comparison.png", dpi=300)
plt.show()


### 9. Export & Publication to Hugging Face Hub
We package the delivered model (`model_partial`) and tokenizer, and generate the **Model Card**.


In [ ]:
from huggingface_hub import login, HfApi

save_dir = "./delivered_model_pos"
model_partial.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
print(f"Delivered model saved to {save_dir}")

# Construct Model Card
model_card_content = f"""---
language: en
license: mit
tags:
- token-classification
- bert
- pos-tagging
- upos
datasets:
- universal_dependencies
metrics:
- accuracy
- f1
---

# BERT-base-uncased for POS Tagging (UD English EWT) - Delivered Model

## Model Description
This model is an adapted `bert-base-uncased` checkpoint for 17-class universal Part-of-Speech (UPOS) tagging on the **Universal Dependencies English EWT** benchmark.

## Subword Handling
- First subwords of each token are assigned the ground-truth UPOS tag.
- Continuation subwords, special tokens ([CLS], [SEP]), and padding are masked with label `-100`.

## Adaptation Comparison: Delivered vs Rejected Alternative
- **Delivered Model:** Partial Fine-Tuning (Top 2 encoder layers + Head with differential learning rates: Head `1e-3`, Encoder `2e-5`).
- **Rejected Alternative:** Feature-Based Adaptation (Frozen BERT body + Linear token classifier).

### Empirical Performance Comparison:
- **Partial Fine-Tuning Test Accuracy:** {test_acc_b:.4f} | **Macro-F1:** {test_f1_b:.4f}
- **Feature-Based Test Accuracy:** {test_acc_a:.4f} | **Macro-F1:** {test_f1_a:.4f}
- **Technical Justification:** Fine-tuning the top 2 encoder layers allows syntax-specific representations in BERT's upper layers to specialize into granular morphological distinctions, improving accuracy while training only ~12% of parameters.

## Training Configuration
- **Base Checkpoint:** `bert-base-uncased` (110M params)
- **Trainable Parameters:** {trainable_params_b:,}
- **Differential Optimizer:** AdamW (Top Layers LR: 2e-5, Head LR: 1e-3)
- **Batch Size:** {batch_size}
- **Epochs:** {epochs}
- **Seed:** 42

## Access Permissions
If private, collaborator access has been granted to user `Dexterg83` per course assignment instructions.
"""

with open(os.path.join(save_dir, "README.md"), "w") as f:
    f.write(model_card_content)
print("Model card README.md created.")

# To push to Hugging Face Hub in Colab:
# login(token="YOUR_HF_TOKEN")
# hf_repo_name = "YOUR_USERNAME/bert-base-uncased-ud-ewt-pos"
# model_partial.push_to_hub(hf_repo_name, private=False)
# tokenizer.push_to_hub(hf_repo_name, private=False)
print("Ready for Hugging Face Hub upload. Uncomment the lines above with your token and repo name.")
